# 연령대별 구간내(Local) AUC 분석

박지은님 리포트의 "고령 구간 FNR 높음"이 진짜 AUC 개선 기회인지, 아니면
"그 연령대는 원래 성공률이 낮아서 임계값 기준으로만 그렇게 보이는 것"인지
확인합니다.

**핵심 질문**: 연령대 "안에서만" 봤을 때도 모델이 순위를 잘 매기고 있는가?
(전체 AUC가 아니라, 그 연령대 환자들끼리만 비교했을 때의 AUC)

가벼운 분석용 노트북이라 몇 초 안에 끝납니다.

## 1. 데이터 로드

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

CACHE_DIR = "blend_cache"
DATA_DIR = "../data"

oof_pred = np.load(f"{CACHE_DIR}/oof_lgbm_tuned.npy")
y = np.load(f"{CACHE_DIR}/oof_y.npy")

train_raw = pd.read_csv(f"{DATA_DIR}/train.csv")
age = train_raw["시술 당시 나이"].values

print(f"행 수 확인: oof={len(oof_pred)}, y={len(y)}, age={len(age)}")
print(f"나이 고유값: {sorted(pd.Series(age).unique())}")

행 수 확인: oof=256351, y=256351, age=256351
나이 고유값: ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세', '알 수 없음']


## 2. 연령대별 Local AUC 계산

In [2]:
results = []
for group in sorted(pd.Series(age).dropna().unique()):
    mask = (age == group)
    n = mask.sum()
    y_g = y[mask]
    pred_g = oof_pred[mask]

    n_pos = int(y_g.sum())
    n_neg = n - n_pos
    pos_rate = y_g.mean()

    if n_pos == 0 or n_neg == 0:
        local_auc = float("nan")
    else:
        local_auc = roc_auc_score(y_g, pred_g)

    results.append({
        "연령대": group,
        "표본수": n,
        "양성수": n_pos,
        "양성률": pos_rate,
        "Local_AUC": local_auc,
    })

result_df = pd.DataFrame(results).sort_values("연령대").reset_index(drop=True)
print(result_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print()
print(f"전체 표본 가중평균 Local AUC: {(result_df['표본수'] * result_df['Local_AUC']).sum() / result_df['표본수'].sum():.4f}")
print(f"(참고용) 전체 AUC: {roc_auc_score(y, oof_pred):.5f}")

    연령대    표본수   양성수    양성률  Local_AUC
만18-34세 102476 33061 0.3226     0.7030
만35-37세  57780 16086 0.2784     0.7010
만38-39세  39247  8522 0.2171     0.7101
만40-42세  37348  5953 0.1594     0.7374
만43-44세  12253  1446 0.1180     0.8293
만45-50세   6918  1160 0.1677     0.8316
 알 수 없음    329     0 0.0000        NaN

전체 표본 가중평균 Local AUC: 0.7172
(참고용) 전체 AUC: 0.73998


## 3. 해석 가이드

- **Local AUC가 다른 연령대보다 눈에 띄게 낮은 그룹**이 있다면, 그 안에서
  모델이 잘 구분 못하고 있다는 뜻 → 진짜 개선 기회일 수 있습니다.
- **표본수가 작은 그룹**(예: 만45-50세)은 Local AUC가 원래 불안정하게
  나올 수 있으니, 양성수가 너무 적으면(예: 50건 미만) 신뢰하기 어렵습니다.
- 모든 그룹이 0.6~0.7대로 비슷하게 나온다면, "이 연령대가 원래 어려운 것"이고
  추가로 팔 만한 구조적 약점은 아닐 가능성이 높습니다.
- 반대로 표본수 충분한 그룹에서 Local AUC가 0.55 이하로 뚝 떨어진다면, 그건
  모델이 그 그룹 안에서 거의 무작위로 찍고 있다는 뜻이라 흥미로운 신호입니다.